# CS6910: Assignment 3 - Task 5: Machine Translation using Transformer Model

## Objective & Architecture Overview
In this notebook, we implement an **English-to-Hindi Machine Translation system** using a **Transformer model with 3 encoder layers and 3 decoder layers** built with PyTorch primitives (`nn.Transformer`, `nn.TransformerEncoderLayer`, `nn.TransformerDecoderLayer`).

### Key Specifications:
1. **Encoder Architecture**: 3 Transformer Encoder layers ($d_{model}=100$, $nhead=4$, $d_{ff}=256$, dropout=0.1).
2. **Decoder Architecture**: 3 Transformer Decoder layers ($d_{model}=100$, $nhead=4$, $d_{ff}=256$, dropout=0.1).
3. **Source Representations (English)**: Pre-trained GloVe 100d word embeddings (`glove.6B.100d.txt`).
4. **Target Representations (Hindi)**: FastText 100d embeddings (trained on the Hindi training corpus using `gensim.models.FastText`).
5. **Tokenization**: Subword BPE (Byte-Pair Encoding) trained via HuggingFace `tokenizers`.
6. **Evaluation Metrics**: BLEU@1, BLEU@2, BLEU@3, and BLEU@4 scores evaluated on Validation and Test sets.
7. **Report Outputs**: Top 5 training examples and top 5 test examples (Source, Ground Truth, Model Prediction).

---
## Mathematical Foundations

### 1. Scaled Dot-Product Attention & Multi-Head Attention
Given Query ($Q$), Key ($K$), and Value ($V$) projections:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

For Multi-Head Attention with $h=4$ heads:
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

### 2. Sinusoidal Positional Encodings
$$\text{PE}_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$\text{PE}_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### 3. Causal & Padding Masking
- **Target Causal Mask**: Prevents the decoder from attending to future tokens $t+1, \dots, T$ during autoregressive target training.
- **Padding Mask**: Suppresses attention weights for `<pad>` tokens in input sequences.


In [1]:
# Cell 1: Imports, Random Seed & Device Setup
import os
import math
import time
import json
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

# Set seeds for deterministic training
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Select compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


ModuleNotFoundError: No module named 'nltk'

In [ ]:
# Cell 2: Load English-Hindi Machine Translation Datasets
train_path = 'DATASETS/Machine translation datasets/team10_hi/team10_hi_train.csv'
valid_path = 'DATASETS/Machine translation datasets/team10_hi/team10_hi_valid.csv'
test_path  = 'DATASETS/Machine translation datasets/team10_hi/team10_hi_test.csv'

df_train = pd.read_csv(train_path)
df_valid = pd.read_csv(valid_path)
df_test  = pd.read_csv(test_path)

print(f"Train samples: {len(df_train):,}")
print(f"Valid samples: {len(df_valid):,}")
print(f"Test samples : {len(df_test):,}")

# Inspect first few samples
print("
Sample Train Pair:")
print("EN (Source):", df_train['source'].iloc[0])
print("HI (Target):", df_train['target'].iloc[0])


In [ ]:
# Cell 3: Subword BPE Tokenizer Training & Helper Functions
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCAB_SIZE_EN = 10000
VOCAB_SIZE_HI = 10000

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[SOS]", "[EOS]"]
PAD_IDX = 0
UNK_IDX = 1
SOS_IDX = 2
EOS_IDX = 3

# 1. Train English BPE Tokenizer
tokenizer_en = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_en.pre_tokenizer = Whitespace()
trainer_en = BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=VOCAB_SIZE_EN)

# Save source texts to temp file for tokenizer training
with open("temp_train_en.txt", "w", encoding="utf-8") as f:
    for text in df_train['source'].astype(str):
        f.write(text.strip() + "\n")

tokenizer_en.train(files=["temp_train_en.txt"], trainer=trainer_en)

# 2. Train Hindi BPE Tokenizer
tokenizer_hi = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer_hi.pre_tokenizer = Whitespace()
trainer_hi = BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=VOCAB_SIZE_HI)

with open("temp_train_hi.txt", "w", encoding="utf-8") as f:
    for text in df_train['target'].astype(str):
        f.write(text.strip() + "\n")

tokenizer_hi.train(files=["temp_train_hi.txt"], trainer=trainer_hi)

# Clean up temp files
os.remove("temp_train_en.txt")
os.remove("temp_train_hi.txt")

src_vocab_size = tokenizer_en.get_vocab_size()
tgt_vocab_size = tokenizer_hi.get_vocab_size()

print(f"English Vocab Size: {src_vocab_size}")
print(f"Hindi Vocab Size  : {tgt_vocab_size}")


In [ ]:
# Cell 4: Pre-trained GloVe 100d Embedding Matrix for English
EMBED_DIM = 100
glove_path = 'DATASETS/glove.6B.100d.txt'

print("Loading GloVe 100d embeddings...")
glove_dict = {}
with open(glove_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(' ')
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove_dict[word] = vector

print(f"Loaded {len(glove_dict):,} GloVe vectors.")

# Initialize English embedding matrix
src_vocab = tokenizer_en.get_vocab()
src_embedding_matrix = np.random.normal(scale=0.1, size=(src_vocab_size, EMBED_DIM)).astype(np.float32)
# Set PAD token to zeros
src_embedding_matrix[PAD_IDX] = np.zeros(EMBED_DIM, dtype=np.float32)

found_count = 0
for token, idx in src_vocab.items():
    # Clean up token BPE prefixes if any
    clean_token = token.lower().lstrip('Ġ').lstrip('##')
    if clean_token in glove_dict:
        src_embedding_matrix[idx] = glove_dict[clean_token]
        found_count += 1

print(f"Matched {found_count}/{src_vocab_size} tokens ({found_count/src_vocab_size:.1%}) with GloVe embeddings.")


In [ ]:
# Cell 5: FastText 100d Embedding Training for Hindi
from gensim.models import FastText

print("Training FastText 100d model on Hindi training corpus...")
# Tokenize Hindi training text into lists of tokens
hindi_sentences = [
    tokenizer_hi.encode(text).tokens 
    for text in df_train['target'].astype(str)
]

fasttext_model = FastText(
    sentences=hindi_sentences, 
    vector_size=EMBED_DIM, 
    window=5, 
    min_count=1, 
    workers=4, 
    epochs=10
)

print("FastText model training complete.")

# Build Hindi embedding matrix
tgt_vocab = tokenizer_hi.get_vocab()
tgt_embedding_matrix = np.random.normal(scale=0.1, size=(tgt_vocab_size, EMBED_DIM)).astype(np.float32)
tgt_embedding_matrix[PAD_IDX] = np.zeros(EMBED_DIM, dtype=np.float32)

ft_found = 0
for token, idx in tgt_vocab.items():
    if token in fasttext_model.wv:
        tgt_embedding_matrix[idx] = fasttext_model.wv[token]
        ft_found += 1

print(f"Populated {ft_found}/{tgt_vocab_size} tokens ({ft_found/tgt_vocab_size:.1%}) with FastText embeddings.")


In [ ]:
# Cell 6: PyTorch TranslationDataset & Dynamic Collate DataLoader
class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer_en, tokenizer_hi, max_len=60):
        self.sources = df['source'].astype(str).tolist()
        self.targets = df['target'].astype(str).tolist()
        self.tokenizer_en = tokenizer_en
        self.tokenizer_hi = tokenizer_hi
        self.max_len = max_len

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        src_text = self.sources[idx]
        tgt_text = self.targets[idx]

        src_ids = [SOS_IDX] + self.tokenizer_en.encode(src_text).ids[:self.max_len - 2] + [EOS_IDX]
        tgt_ids = [SOS_IDX] + self.tokenizer_hi.encode(tgt_text).ids[:self.max_len - 2] + [EOS_IDX]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long), src_text, tgt_text

def pad_collate_fn(batch):
    src_batch, tgt_batch, src_texts, tgt_texts = zip(*batch)

    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)

    return src_padded, tgt_padded, src_texts, tgt_texts

BATCH_SIZE = 64

train_dataset = TranslationDataset(df_train, tokenizer_en, tokenizer_hi)
valid_dataset = TranslationDataset(df_valid, tokenizer_en, tokenizer_hi)
test_dataset  = TranslationDataset(df_test, tokenizer_en, tokenizer_hi)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate_fn)

print(f"Train Batches: {len(train_loader)}")
print(f"Valid Batches: {len(valid_loader)}")


In [ ]:
# Cell 7: Positional Encoding Module
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # (1, max_len, d_model) for batch_first=True

        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


In [ ]:
# Cell 8: Seq2Seq Transformer Model (3 Layers Encoder & 3 Layers Decoder)
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=100, nhead=4, 
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=256, 
                 dropout=0.1, src_embed_matrix=None, tgt_embed_matrix=None):
        super(Seq2SeqTransformer, self).__init__()

        self.d_model = d_model

        # Source & Target Embeddings
        self.src_tok_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        if src_embed_matrix is not None:
            self.src_tok_emb.weight.data.copy_(torch.tensor(src_embed_matrix))

        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        if tgt_embed_matrix is not None:
            self.tgt_tok_emb.weight.data.copy_(torch.tensor(tgt_embed_matrix))

        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)

        # Transformer Primitive with 3 Encoder & 3 Decoder Layers
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        # Output Generator Linear Layer
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, 
                src_padding_mask=None, tgt_padding_mask=None):
        # Scale embeddings by sqrt(d_model)
        src_emb = self.positional_encoding(self.src_tok_emb(src) * math.sqrt(self.d_model))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(tgt) * math.sqrt(self.d_model))

        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        return self.generator(outs)


In [ ]:
# Cell 9: Causal Masking & Key Padding Mask Functions
def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def create_mask(src, tgt, device):
    src_seq_len = src.shape[1]
    tgt_seq_len = tgt.shape[1]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len, device)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).type(torch.bool)

    src_padding_mask = (src == PAD_IDX)
    tgt_padding_mask = (tgt == PAD_IDX)

    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


In [ ]:
# Cell 10: Training & Validation Epoch Loop Functions
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for src, tgt, _, _ in dataloader:
        src = src.to(device)
        tgt = tgt.to(device)

        # Target input (shifted right) and Target expected output
        tgt_input = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input, device)

        optimizer.zero_grad()
        logits = model(
            src=src,
            tgt=tgt_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask
        )

        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate_loss(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for src, tgt, _, _ in dataloader:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input, device)

            logits = model(
                src=src,
                tgt=tgt_input,
                src_mask=src_mask,
                tgt_mask=tgt_mask,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask
            )

            loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
            total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# Cell 11: Instantiate Model & Run Training Loop
model = Seq2SeqTransformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=100,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=256,
    dropout=0.1,
    src_embed_matrix=src_embedding_matrix,
    tgt_embed_matrix=tgt_embedding_matrix
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

EPOCHS = 10
best_val_loss = float('inf')

print("Starting Transformer Training (3 Encoder & 3 Decoder Layers)...")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate_loss(model, valid_loader, criterion, device)
    elapsed = time.time() - start_time

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'transformer_task5_best.pt')
        saved_str = "[SAVED BEST]"
    else:
        saved_str = ""

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s {saved_str}")

# Load best checkpoint
model.load_state_dict(torch.load('transformer_task5_best.pt'))
print("Best checkpoint loaded successfully.")


In [ ]:
# Cell 12: Autoregressive Translation Decoding (Greedy Search)
def translate_sentence(model, src_sentence, tokenizer_en, tokenizer_hi, device, max_len=60):
    model.eval()

    # Tokenize source
    src_ids = [SOS_IDX] + tokenizer_en.encode(src_sentence).ids[:max_len - 2] + [EOS_IDX]
    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)
    src_mask = torch.zeros((1, src_tensor.shape[1]), device=device).type(torch.bool)

    # Encode source tokens
    with torch.no_grad():
        src_emb = model.positional_encoding(model.src_tok_emb(src_tensor) * math.sqrt(model.d_model))
        memory = model.transformer.encoder(src_emb)

    tgt_ids = [SOS_IDX]

    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_ids, dtype=torch.long).unsqueeze(0).to(device)
        tgt_mask = generate_square_subsequent_mask(tgt_tensor.shape[1], device)

        with torch.no_grad():
            tgt_emb = model.positional_encoding(model.tgt_tok_emb(tgt_tensor) * math.sqrt(model.d_model))
            outs = model.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask)
            logits = model.generator(outs[:, -1, :])
            next_token = logits.argmax(dim=-1).item()

        if next_token == EOS_IDX:
            break

        tgt_ids.append(next_token)

    # Decode target IDs to string
    translated_tokens = tokenizer_hi.decode(tgt_ids[1:])
    return translated_tokens


In [ ]:
# Cell 13: Compute BLEU@k (k=1, 2, 3, 4) Scores on Test Set
def evaluate_bleu_scores(model, df_eval, tokenizer_en, tokenizer_hi, device, sample_limit=500):
    print(f"Evaluating BLEU scores on {min(sample_limit, len(df_eval))} samples...")

    references = []
    hypotheses = []

    smooth_fn = SmoothingFunction().method1

    for i in range(min(sample_limit, len(df_eval))):
        src_text = str(df_eval['source'].iloc[i])
        gt_text  = str(df_eval['target'].iloc[i])

        pred_text = translate_sentence(model, src_text, tokenizer_en, tokenizer_hi, device)

        # Tokenize target and prediction for BLEU
        gt_tokens = tokenizer_hi.encode(gt_text).tokens
        pred_tokens = tokenizer_hi.encode(pred_text).tokens

        references.append([gt_tokens])
        hypotheses.append(pred_tokens)

    bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smooth_fn)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0.0, 0.0), smoothing_function=smooth_fn)
    bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0.0), smoothing_function=smooth_fn)
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth_fn)

    bleu_df = pd.DataFrame({
        'Metric': ['BLEU@1', 'BLEU@2', 'BLEU@3', 'BLEU@4'],
        'Score': [bleu1 * 100, bleu2 * 100, bleu3 * 100, bleu4 * 100]
    })

    print("
--- Evaluation BLEU Scores ---")
    print(bleu_df.to_string(index=False))
    return bleu_df

bleu_results = evaluate_bleu_scores(model, df_test, tokenizer_en, tokenizer_hi, device, sample_limit=500)


In [ ]:
# Cell 14: Extract 5 Training and 5 Test Predictions with Ground Truth for Report
def extract_report_examples(model, df_train, df_test, tokenizer_en, tokenizer_hi, device):
    print("=" * 80)
    print(" 5 TRAINING EXAMPLES & TRANSLATION OUTPUTS ")
    print("=" * 80)

    train_samples = df_train.sample(5, random_state=42).reset_index(drop=True)
    for i in range(5):
        src = str(train_samples['source'].iloc[i])
        gt  = str(train_samples['target'].iloc[i])
        pred = translate_sentence(model, src, tokenizer_en, tokenizer_hi, device)
        print(f"\nSample {i+1}:")
        print(f"  Source (EN)      : {src}")
        print(f"  Ground Truth (HI): {gt}")
        print(f"  Prediction (HI)  : {pred}")

    print("\n" + "=" * 80)
    print(" 5 TEST EXAMPLES & TRANSLATION OUTPUTS ")
    print("=" * 80)

    test_samples = df_test.sample(5, random_state=42).reset_index(drop=True)
    for i in range(5):
        src = str(test_samples['source'].iloc[i])
        gt  = str(test_samples['target'].iloc[i])
        pred = translate_sentence(model, src, tokenizer_en, tokenizer_hi, device)
        print(f"\nSample {i+1}:")
        print(f"  Source (EN)      : {src}")
        print(f"  Ground Truth (HI): {gt}")
        print(f"  Prediction (HI)  : {pred}")

extract_report_examples(model, df_train, df_test, tokenizer_en, tokenizer_hi, device)
